# Phase 3b: evaluate the fine-tuned whisper-medium on dev

Runs the fine-tuned medium (`best/` from notebook 05) through exactly the same evaluation as small in
notebook 04: `cyasr.run_baseline`, the 280 full-length dev clips, long-form sequential decoding, same scoring.
Decoding is fallback, with and without the no-speech rule (`no_speech_threshold 0.6`), which notebook 04
showed removes most invented tails on silent windows. Zero-shot medium is run with the rule too, so the
zero-shot vs fine-tuned comparison uses identical decoding. Greedy is skipped to save compute.

Test is not touched here; it is evaluated once, at the end of the project.

Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to
`MyDrive/cyasr/`, replacing the old one (it must contain `configs/eval_whisper-medium_lora.yaml`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'
CONFIG = 'configs/eval_whisper-medium_lora.yaml'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls {CONFIG}  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt
# Colab ships an old torchao that peft rejects ("only versions above 0.16.0"); we don't use it
!pip uninstall -y -q torchao

In [ ]:
# The fine-tuned model saved by notebook 05 (LoRA already merged, so it is a plain Whisper model)
import yaml
MODEL = yaml.safe_load(open(CONFIG))['model']
!ls -lh {MODEL}

In [ ]:
# Smoke test: 4 dev clips, fallback + rule, saved locally. Checks that the model loads and writes Cypriot text.
import json
!python -m cyasr.run_baseline --config {CONFIG} --decoding fallback --no-speech-threshold 0.6 --max-samples 4
for line in open('runs/whisper-medium_lora_fallback_ns0.6_dev_n4/predictions.jsonl', encoding='utf-8'):
    r = json.loads(line)
    print(f"--- {r['audio_id']} ({r['duration_s']:.0f} s)\nREF: {r['ref'][:200]}\nHYP: {r['hyp'][:200]}")

In [ ]:
# The real runs, all dev clips, written straight to Drive. Rough guess: 35-45 min on an L4.
# (Zero-shot medium with fallback and no rule already exists from the baseline phase.)
import time
runs = [(CONFIG, ''),                                          # fine-tuned, rule off
        (CONFIG, '--no-speech-threshold 0.6'),                 # fine-tuned, rule on
        ('configs/baseline_whisper-medium.yaml', '--no-speech-threshold 0.6')]  # zero-shot, rule on
for cfg_path, extra in runs:
    cfg = yaml.safe_load(open(cfg_path))
    cfg.update(splits=['dev'], output_dir=DRIVE_RUNS)  # the baseline config also lists test; keep it untouched
    yaml.safe_dump(cfg, open('/content/eval.yaml', 'w'), allow_unicode=True)
    t0 = time.time()
    !python -m cyasr.run_baseline --config /content/eval.yaml --decoding fallback {extra}
    print(f'>>> {cfg_path} {extra}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Zero-shot vs fine-tuned, small and medium, fallback decoding, rule off and on. Same dev clips and scoring.
import os
def load(name):  # None if the run has not finished (no metrics.json)
    path = f'{DRIVE_RUNS}/{name}/metrics.json'
    return json.load(open(path)) if os.path.exists(path) else None

print(f"{'size':7} {'model':11} {'rule':4} {'WER basic':>9} {'WER nodiac':>10} {'CER nodiac':>10} {'halluc.':>7} {'empty':>5}")
for size in ['small', 'medium']:
    for label, prefix in [('zero-shot', f'baseline_whisper-{size}'), ('fine-tuned', f'whisper-{size}_lora')]:
        for rule, suffix in [('off', ''), ('0.6', '_ns0.6')]:
            m = load(f'{prefix}_fallback{suffix}_dev')
            if m is None:
                print(f"{size:7} {label:11} {rule:4} {'missing':>9}"); continue
            lv = m['levels']
            print(f"{size:7} {label:11} {rule:4} {lv['basic']['wer']:9.1%} {lv['nodiac']['wer']:10.1%} "
                  f"{lv['nodiac']['cer']:10.1%} {m['hallucinations']:7} {m['empty_hypotheses']:5}")

In [ ]:
# WER (nodiac) overall and by clip length: does medium also invent text on the silent tails of 30-33 s clips?
import jiwer
from cyasr.normalize import normalize

def bucket(d):
    return '<=30s' if d <= 30 else ('30-33s' if d <= 33 else '>33s')

runs = [('zero-shot', 'off', 'baseline_whisper-medium_fallback_dev'),
        ('zero-shot', '0.6', 'baseline_whisper-medium_fallback_ns0.6_dev'),
        ('fine-tuned', 'off', 'whisper-medium_lora_fallback_dev'),
        ('fine-tuned', '0.6', 'whisper-medium_lora_fallback_ns0.6_dev')]
print(f"{'model':11} {'rule':4} {'all':>6} {'<=30s':>6} {'30-33s':>7} {'>33s':>6} {'insert.':>7} {'halluc.':>7}")
import os
for model, rule, name in runs:
    if not os.path.exists(f'{DRIVE_RUNS}/{name}/metrics.json'):  # run not finished
        print(f"{model:11} {rule:4} {'missing':>6}"); continue
    recs = [json.loads(l) for l in open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8')]
    errs, words, ins, halluc = {}, {}, 0, 0
    for r in recs:
        ref, hyp = normalize(r['ref'], 'nodiac'), normalize(r['hyp'], 'nodiac')
        o = jiwer.process_words(ref, hyp)
        for b in (bucket(r['duration_s']), 'all'):
            errs[b] = errs.get(b, 0) + o.substitutions + o.deletions + o.insertions
            words[b] = words.get(b, 0) + len(ref.split())
        ins += o.insertions
        halluc += len(hyp.split()) > 2 * max(len(ref.split()), 1)
    w = {b: errs[b] / words[b] for b in errs}
    print(f"{model:11} {rule:4} {w['all']:6.1%} {w['<=30s']:6.1%} {w['30-33s']:7.1%} {w['>33s']:6.1%} {ins:7} {halluc:7}")

In [ ]:

# Five clips side by side (fallback + rule): reference, zero-shot medium, fine-tuned small, fine-tuned medium
# (zero-shot medium falls back to the run without the rule if the rule run is missing)
import os
import random
def preds(name):
    return {r['audio_id']: r for r in map(json.loads, open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8'))}

zs_name = 'baseline_whisper-medium_fallback_ns0.6_dev'
if not os.path.exists(f'{DRIVE_RUNS}/{zs_name}/metrics.json'):
    zs_name = 'baseline_whisper-medium_fallback_dev'
print('zero-shot run:', zs_name)
zs = preds(zs_name)
sm = preds('whisper-small_lora_fallback_ns0.6_dev')
md_ = preds('whisper-medium_lora_fallback_ns0.6_dev')
short = sorted(a for a in md_ if md_[a]['duration_s'] <= 20)  # short clips are easier to read
for a in random.Random(0).sample(short, 5):
    print(f"--- {a} ({md_[a]['duration_s']:.0f} s)")
    print('REF        :', md_[a]['ref'])
    print('ZERO-SHOT M:', zs[a]['hyp'])
    print('FT SMALL   :', sm[a]['hyp'])
    print('FT MEDIUM  :', md_[a]['hyp'])